# Train a food-photo calorie estimator with FastAI

This beginner-friendly notebook fine-tunes an ImageNet-pretrained ResNet-18 to regress **total calories for a pictured plate**. The app shows calorie estimates only; it never displays a dish name.

The starter data is the public [Nutrition5k FoodSeg103-filtered dataset](https://huggingface.co/datasets/arunapb/nutrition5k-foodseg103), derived from Google's [Nutrition5k](https://github.com/google-research-datasets/Nutrition5k). Nutrition5k includes overhead plate photos and measured plate calories/mass, but was collected in California cafeterias. This filtered starter subset is under 1,000 plates and is not Indian-food-specific. This notebook cannot establish reliable accuracy for Indian dishes or arbitrary meals.

A single RGB photo cannot reveal exact ingredients, hidden oil, recipe, depth, or food weight. Treat all outputs as rough estimates, not medical or dietary advice. The dataset's original license is CC BY 4.0; retain attribution and verify terms before redistributing data or weights. Paper: Thames et al., CVPR 2021, [arXiv:2103.03375](https://arxiv.org/abs/2103.03375).


## 1. Install the training dependencies

Run this once in the project's virtual environment. If Jupyter asks for a kernel, select `food-calorie-estimator` after registering it from the README instructions.

In [ ]:
from pathlib import Path
requirements_path = Path.cwd() / "requirements-training.txt"
if not requirements_path.is_file():
    requirements_path = Path.cwd().parent / "requirements-training.txt"
if not requirements_path.is_file():
    raise FileNotFoundError("Could not find requirements-training.txt in the project folder.")
%pip install -r {requirements_path}

## 2. Imports and repeatable randomness

In [ ]:
import os
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
import fastai
from fastai.vision.all import *
from sklearn.metrics import mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED, reproducible=True)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "nutrition5k-foodseg103"
MODEL_DIR = PROJECT_DIR / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_DIR.resolve())
print("FastAI:", fastai.__version__)
print("Torch:", torch.__version__)


## 3. Download the small public dataset subset

Hugging Face Datasets downloads the data into its local cache. The source split is preserved so we can keep the official test plates out of training and validation.

In [ ]:
DATASET_ID = "arunapb/nutrition5k-foodseg103"
DATASET_REVISION = "cd90be60e10ed039b716cd26b8ba1ecc6ed1e4fc"
dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
print(dataset)
print("Available columns:", dataset["train"].column_names)
print("Train plates:", len(dataset["train"]))
print("Test plates:", len(dataset["test"]))

## 4. Save images and calorie targets locally

The target is `total_calories`: measured kcal for the complete photographed plate, not kcal per 100 g. The dish name/ingredient metadata is not passed to the model.

In [ ]:
rows = []
for split_name in ("train", "test"):
    for row in dataset[split_name]:
        image = row["image"].convert("RGB")
        image_path = DATA_DIR / f"{split_name}_{row['id']}.jpg"
        if not image_path.exists():
            image.save(image_path, format="JPEG", quality=95)
        calories = float(row["total_calories"])
        if not np.isfinite(calories) or calories < 0:
            raise ValueError(f"Invalid calorie target for plate {row['id']}: {calories}")
        rows.append({"image_path": str(image_path), "calories": calories, "source_split": split_name})

df = pd.DataFrame(rows)
train_df = df.loc[df["source_split"] == "train"].reset_index(drop=True)
test_df = df.loc[df["source_split"] == "test"].reset_index(drop=True)
print(f"Training split: {len(train_df)} plates; held-out test split: {len(test_df)} plates")
train_df["calories"].describe()

## 5. Create FastAI data loaders

`RegressionBlock` tells FastAI that calories are numeric values. `RandomSplitter` creates a validation portion only from the source training split; the source test split stays untouched until final evaluation.

In [ ]:
food_block = DataBlock(
    blocks=(ImageBlock, RegressionBlock),
    get_x=ColReader("image_path"),
    get_y=ColReader("calories"),
    splitter=RandomSplitter(valid_pct=0.2, seed=SEED),
    item_tfms=Resize(224),
    batch_tfms=aug_transforms(size=224),
)
dls = food_block.dataloaders(train_df, bs=16, num_workers=0)
dls.show_batch(max_n=6, figsize=(8, 6))

## 6. Fine-tune a pretrained model

The ImageNet-pretrained ResNet-18 starts with general visual features; FastAI replaces its final layer with a numeric regression head. MAE (mean absolute error) is shown in kcal, which is easier to interpret than squared error. CPU-only training may take a while.

In [ ]:
learn = vision_learner(
    dls,
    resnet18,
    loss_func=MSELossFlat(),
    metrics=mae,
)
learn.fine_tune(8, base_lr=2e-3)

## 7. Measure held-out error and build a rough range

First fit a simple linear calibration using validation predictions only, and compute the 90th percentile of absolute calibrated validation errors. The app uses that as a broad, validation-derived range around each estimate; it is not a guarantee for any individual image. Then evaluate once on the untouched official test split and compare against a baseline that always predicts the training-set average. The test split is for reporting only; do not tune using it.

In [ ]:
valid_preds, valid_targets = learn.get_preds(dl=dls.valid)
raw_valid_predictions_kcal = valid_preds.reshape(-1).cpu().numpy()
valid_targets_kcal = valid_targets.reshape(-1).cpu().numpy()
calibration_slope, calibration_intercept = np.polyfit(raw_valid_predictions_kcal, valid_targets_kcal, deg=1)
if not np.isfinite([calibration_slope, calibration_intercept]).all():
    raise ValueError("Validation calibration produced non-finite coefficients.")
valid_predictions_kcal = np.maximum(0, calibration_slope * raw_valid_predictions_kcal + calibration_intercept)
validation_mae_kcal = float(mean_absolute_error(valid_targets_kcal, valid_predictions_kcal))
validation_error_p90_kcal = float(np.quantile(np.abs(valid_targets_kcal - valid_predictions_kcal), 0.90))

test_dl = learn.dls.test_dl(test_df, with_labels=True)
test_preds, test_targets = learn.get_preds(dl=test_dl)
raw_test_predictions_kcal = test_preds.reshape(-1).cpu().numpy()
test_predictions_kcal = np.maximum(0, calibration_slope * raw_test_predictions_kcal + calibration_intercept)
test_targets_kcal = test_targets.reshape(-1).cpu().numpy()
test_mae_kcal = float(mean_absolute_error(test_targets_kcal, test_predictions_kcal))
training_mean_kcal = float(train_df["calories"].mean())
baseline_test_mae_kcal = float(mean_absolute_error(test_targets_kcal, np.full_like(test_targets_kcal, training_mean_kcal)))

print(f"Validation MAE: {validation_mae_kcal:.0f} kcal per plate")
print(f"Validation 90th-percentile absolute error: {validation_error_p90_kcal:.0f} kcal")
print(f"Official held-out test MAE: {test_mae_kcal:.0f} kcal per plate")
print(f"Training-mean baseline test MAE: {baseline_test_mae_kcal:.0f} kcal per plate")
print("These results apply to this small California-cafeteria dataset, not specifically to Indian meals.")

## 8. Export the trained model and uncertainty metadata

The app loads the TorchScript model and calibration metadata from `models/`. The FastAI `.pkl` export is retained as the editable learner checkpoint; the app does not deserialize it at runtime.

In [ ]:
model_path = MODEL_DIR / "food_calorie_model.pkl"
deployment_model_path = MODEL_DIR / "food_calorie_model.ts"
metrics_path = MODEL_DIR / "calorie_metrics.json"
learn.export(model_path)
learn.model.eval()
example_input = torch.zeros((1, 3, 224, 224), dtype=torch.float32)
deployment_model = torch.jit.trace(learn.model.cpu(), example_input)
deployment_model.save(str(deployment_model_path))
metrics = {
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "dataset_license": "CC BY 4.0 (verify original source terms before redistribution)",
    "target": "total calories for one pictured plate, kcal",
    "validation_mae_kcal": validation_mae_kcal,
    "validation_absolute_error_p90_kcal": validation_error_p90_kcal,
    "held_out_test_mae_kcal": test_mae_kcal,
    "training_mean_baseline_test_mae_kcal": baseline_test_mae_kcal,
    "calibration_slope": float(calibration_slope),
    "calibration_intercept_kcal": float(calibration_intercept),
    "cuisine_coverage_note": "Nutrition5k was collected in California cafeterias; Indian-food accuracy is not established.",
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Exported:", model_path.resolve())
print("Exported deployment model:", deployment_model_path.resolve())
print("Exported:", metrics_path.resolve())

## 9. Run the calorie-only app

Open a terminal in the `food-calorie-estimator` project directory and run `streamlit run app.py`. The app reports only calories and uncertainty, never the dish name. If you collect a new Indian-food dataset, include measured serving weights and recipe-based calorie targets and evaluate on plates not seen during training.